# 기상청_단기예보 조회서비스
- 이름: `VilageFcstInfoService_2.0`
- 링크: [`https://www.data.go.kr/data/15084084/openapi.do?`](https://www.data.go.kr/data/15084084/openapi.do?)
- 참고 문서: [로컬 파일](../../resource/datasets//02_VilageFcstInfoService_2/)
- 라이센스: **명시 필요**

In [1]:
from dotenv import load_dotenv
import os
import pprint
import requests
import pandas as pd

load_dotenv()

endpoint="https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0"
API_KEY = os.getenv("VILAGE_FCST_INFO_SERVICE_API_KEY")

# 상세기능정보 (실황 정보 - 오늘자)

In [13]:
from datetime import datetime

TODAY = str(datetime.now().date()).replace("-", "")

In [43]:
# 상세기능정보
url = f"{endpoint}/getUltraSrtNcst"

# 오늘 YYYYMMDD 뽑기


params = {
      "serviceKey": API_KEY,
      "numOfRows": 100, # 페이지당 결과 수 
      "pageNo": 1,        # 
      "dataType": "JSON", # XML 또는 JSON
      "base_date": TODAY,  # 최근 날짜
      "base_time": "0000",      # 발표 시각 (HH00)
      "nx": 55,         # 기상청 격자 x좌표
      "ny": 127,        # 기상청 격자 y좌표
  }

res = requests.get(url, params=params, timeout=10)
res.encoding = "utf-8"

# print(res.status_code)
# print(data := res.json())
data = res.json()
# pprint.pprint(short_predict := data.get("response").get("body").get("items").get("item"))
short_predict = data.get("response").get("body").get("items").get("item")

short_predict_df = pd.DataFrame(short_predict, columns=list(short_predict[0].keys()))
short_predict_df.head()

,baseDate,baseTime,category,nx,ny,obsrValue
0,20260705,0000,PTY,55,127,0
1,20260705,0000,REH,55,127,85
2,20260705,0000,RN1,55,127,0
3,20260705,0000,T1H,55,127,23.7
4,20260705,0000,UUU,55,127,0.2


In [36]:
# 카테고리 읽기 쉽게 바꾸기
CAT_MAP = {
  "T1H": "기온 (도)",
  "RN1": "1시간 강수량 (mm)",
  "UUU": "동서 바람 성분 (m/s)",
  "VVV": "남북 바람 성분 (m/s)",
  "REH": "습도 (%)",
  "PTY": "강수 형태",
  "VEC": "풍향 (deg)",
  "WSD": "풍속 (m/s)",
}

PTY_MAP = {
  "0": "없음", 
  "1": "비", 
  "2": "비/눈", 
  "3": "눈", 
  "5": "빗방울", 
  "6": "빗방울·눈날림", 
  "7": "눈날림"
}

short_predict_df_korean = short_predict_df.copy()

short_predict_df_korean.loc[(short_predict_df_korean["category"] == "PTY"), "obsrValue"] \
  = short_predict_df_korean[short_predict_df_korean["category"] == "PTY"]["obsrValue"] \
  .map(PTY_MAP)

short_predict_df_korean["category"] = list(map(lambda x: CAT_MAP.get(x, x), short_predict_df_korean["category"]))
# category: 
  # - T1H: 기온 (도)
  # - RN1: 1시간 강수량 (mm)
  # - UUU: 동서 바람 성분 (m/s)
  # - VVV: 남북 바람 성분 (m/s)
  # - REH: 습도 (%)
  # - PTY 강수 형태 (0=없음, 1=비, 2=비/눈, 3=눈, 5=빗방울, 6=빗방울·눈날림, 7=눈날림)
  # - VEC: 풍향 (deg)
  # - WSD: 풍속 (m/s)
# short_predict_df_korean.info()
short_predict_df_korean

,baseDate,baseTime,category,nx,ny,obsrValue
0,20260705,0000,강수 형태,55,127,없음
1,20260705,0000,습도 (%),55,127,85
2,20260705,0000,1시간 강수량 (mm),55,127,0
3,20260705,0000,기온 (도),55,127,23.7
4,20260705,0000,동서 바람 성분 (m/s),55,127,0.2
5,20260705,0000,풍향 (deg),55,127,190
6,20260705,0000,남북 바람 성분 (m/s),55,127,1.1
7,20260705,0000,풍속 (m/s),55,127,1.1


# 초단기예보조회

In [51]:
# 현재 HH00
print(datetime.now())
print(str(datetime.now().hour) + "00")

2026-07-05 11:22:37.888118
1100


In [52]:
# 초단기 예보
url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtFcst"

params = {
  "serviceKey": API_KEY,
  "numOfRows": 100,
  "pageNo": 1,
  "dataType": "JSON",
  "base_date": TODAY,
  "base_time": str(datetime.now().hour) + "00",
  "nx": 55,
  "ny": 127
}

res = requests.get(url, params=params, timeout=10)
res.encoding = "utf-8"
# 
# print(res.status_code)
# print(data := res.json())
data = res.json()
# pprint.pprint(short_predict := data.get("response").get("body").get("items").get("item"))
short_predict = data.get("response").get("body").get("items").get("item")

getUltraSrtFcst_df = pd.DataFrame(short_predict, columns=list(short_predict[0].keys()))

getUltraSrtFcst_df.head()

,baseDate,baseTime,category,fcstDate,fcstTime,fcstValue,nx,ny
0,20260705,1130,LGT,20260705,1200,0,55,127
1,20260705,1130,LGT,20260705,1300,0,55,127
2,20260705,1130,LGT,20260705,1400,0,55,127
3,20260705,1130,LGT,20260705,1500,0,55,127
4,20260705,1130,LGT,20260705,1600,0,55,127


In [53]:
# category: 
  # - T1H: 기온 (도)
  # - RN1: 1시간 강수량 (mm)
  # - UUU: 동서 바람 성분 (m/s)
  # - VVV: 남북 바람 성분 (m/s)
  # - REH: 습도 (%)
  # - PTY 강수 형태 (0=없음, 1=비, 2=비/눈, 3=눈, 5=빗방울, 6=빗방울·눈날림, 7=눈날림)
  # - VEC: 풍향 (deg)
  # - WSD: 풍속 (m/s)
  # - POP: 강수 가능성 (%)
  # - PTY: 강수 종류 (1 = 비)
  # - RN1: 1시간 예상 강수량 (mm)
  # - T1H: 기온	(℃)
	# - RN1: 1시간 강수량	범주 (1 mm)
	# - SKY: 하늘상태	코드값
	# - UUU: 동서바람성분	(m/s)
	# - VVV: 남북바람성분	(m/s)
	# - REH: 습도	(%)
	# - PTY: 강수형태	코드값
	# - POP: 강수확률	(%)
	# - LGT: 낙뢰	(kA)(킬로암페어)
	# - VEC: 풍향	(deg)
	# - WSD: 풍속	(m/s)	

CAT_MAP = {
  "T1H": "기온 (도)",
  "RN1": "1시간 강수량 (mm)",
  "UUU": "동서 바람 성분 (m/s)",
  "VVV": "남북 바람 성분 (m/s)",
  "REH": "습도 (%)",
  "PTY": "강수 형태 (0=없음, 1=비, 2=비/눈, 3=눈, 5=빗방울, 6=빗방울·눈날림, 7=눈날림)",
  "VEC": "풍향 (deg)",
  "WSD": "풍속 (m/s)",
  "POP": "강수 가능성 (%)",
  "PTY": "강수 종류 (1 = 비)",
  "RN1": "1시간 예상 강수량 (mm)",
  "T1H": "기온 (℃)",
  "RN1": "1시간 강수량 범주 (1 mm)",
  "SKY": "하늘상태 코드값",
  "UUU": "동서바람성분 (m/s)",
  "VVV": "남북바람성분 (m/s)",
  "REH": "습도 (%)",
  "PTY": "강수형태 코드값",
  "POP": "강수확률 (%)",
  "LGT": "낙뢰 (kA)(킬로암페어)",
  "VEC": "풍향 (deg)",
  "WSD": "풍속 (m/s)	",
}

PTY_MAP = {
  "0": "없음", 
  "1": "비", 
  "2": "비/눈", 
  "3": "눈", 
  "5": "빗방울", 
  "6": "빗방울·눈날림", 
  "7": "눈날림"
}




getUltraSrtFcst_df_korean = getUltraSrtFcst_df.copy()

# getUltraSrtFcst_df_korean.loc[(getUltraSrtFcst_df_korean["category"] == "PTY"), "obsrValue"] \
#   = getUltraSrtFcst_df_korean.loc[(getUltraSrtFcst_df_korean["category"] == "PTY"), "obsrValue"] \
#   .map(PTY_MAP)

getUltraSrtFcst_df_korean["category"] = list(map(lambda x: CAT_MAP.get(x, x), getUltraSrtFcst_df_korean["category"]))

getUltraSrtFcst_df_korean
# 항목명(영문)	항목명(국문)	항목크기	항목구분	샘플데이터	항목설명
# numOfRows	  한 페이지 결과 수	4	    1	        50	한 페이지당 표출 데이터 수
# pageNo	페이지 번호	4	1	1	페이지 수
# totalCount	데이터 총 개수	10	1	1	데이터 총 개수
# resultCode	응답메시지 코드	2	1	00	응답 메시지코드
# resultMsg	응답메시지 내용	100	1	NORMAL SERVICE	응답 메시지 설명
# dataType	데이터 타입	4	1	XML	응답자료형식 (XML/JSON)
# baseDate	발표일자	8	1	20210628	‘21년 6월 28일 발표
# baseTime	발표시각	6	1	0500	05시 발표
# fcstDate	예보일자	8	1	20210628	‘21년 6월 28일 예보
# fcstTime	예보시각	4	1	0600	6시 예보
# category	자료구분문자	3	1	TMP	자료구분코드 
# fcstValue	예보 값	2	1	21
# nx	예보지점 X 좌표	2	1	55	입력한 예보지점 X 좌표
# ny	예보지점 Y 좌표	2	1	127	입력한 예보지점 Y 좌표


,baseDate,baseTime,category,fcstDate,fcstTime,fcstValue,nx,ny
0,20260705,1130,낙뢰 (kA)(킬로암페어),20260705,1200,0,55,127
1,20260705,1130,낙뢰 (kA)(킬로암페어),20260705,1300,0,55,127
2,20260705,1130,낙뢰 (kA)(킬로암페어),20260705,1400,0,55,127
3,20260705,1130,낙뢰 (kA)(킬로암페어),20260705,1500,0,55,127
4,20260705,1130,낙뢰 (kA)(킬로암페어),20260705,1600,0,55,127
...,...,...,...,...,...,...,...,...
61,20260705,1130,강수확률 (%),20260705,1300,30,55,127
62,20260705,1130,강수확률 (%),20260705,1400,30,55,127
63,20260705,1130,강수확률 (%),20260705,1500,30,55,127
64,20260705,1130,강수확률 (%),20260705,1600,30,55,127
